# Financial News Sentiment Lab: results report

Can a fine-tuned transformer, or a small LLM fine-tuned with LoRA, beat a strong bag-of-words
baseline at labelling financial news tweets as **Bearish**, **Bullish** or **Neutral**, and is
the difference statistically significant?

This notebook is the public summary of the GitHub project
[fin-news-sentiment-lab](https://github.com/samarthkala5/fin-news-sentiment-lab).
**It trains nothing.** Every number and figure below is read from files committed in that
repository (`data/splits/` and `results/`), which the project's scripts wrote. A section whose
results are not committed yet prints *not run yet*.

| Part | Method | Libraries |
|---|---|---|
| Baseline | TF-IDF features + logistic regression | scikit-learn |
| Neural baseline | BiLSTM over learned word embeddings | TensorFlow / Keras |
| Transformer | DistilBERT fine-tuned with an explicit training loop | PyTorch, Hugging Face Transformers |
| LLM | small instruct model, zero-shot and LoRA-fine-tuned | Unsloth, PEFT, TRL |
| Statistics | bootstrap 95% CIs, McNemar's test with Holm correction | NumPy, statsmodels |
| Extras | K-Means on sentence embeddings with a NumPy PCA; RAG over central-bank statements | scikit-learn, sentence-transformers, LangChain, FAISS |

## Setup

On Kaggle, switch **Internet on** (Settings panel) so that the next cell can clone the
repository. Locally, open the notebook from the repo root or from `notebooks/`; then nothing is
downloaded. Only light packages are needed (pandas, matplotlib, scikit-learn), and Kaggle
already has them.

In [ ]:
import os
import sys

REPO_URL = "https://github.com/samarthkala5/fin-news-sentiment-lab.git"
BRANCH = "main"

if os.path.exists("../src/fnsl/config.py"):  # opened from notebooks/ in a local checkout
    os.chdir("..")
if not os.path.exists("src/fnsl/config.py"):  # e.g. on Kaggle: fetch the repository first
    if not os.path.isdir("fin-news-sentiment-lab"):
        !git clone --depth 1 -b $BRANCH $REPO_URL
    os.chdir("fin-news-sentiment-lab")
sys.path.insert(0, os.path.abspath("src"))  # lets `import fnsl` work without pip install

Imports, chart colours and a small helper that reads one results file. A file written by a
`--smoke` run (tiny data, only used to check that the code works) is treated as missing.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

from fnsl import config, data, metrics, reporting

print("Repository commit:", metrics.git_commit())

# Chart colours: a fixed categorical order (blue, orange, aqua) and quiet greys for the rest.
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "text.color": INK_2,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "legend.frameon": False,
    }
)


def read_result(path):
    """A results JSON as a dict, or None if it is missing or was written by a --smoke run."""
    if not path.exists():
        return None
    record = json.loads(path.read_text(encoding="utf-8"))
    return None if record.get("smoke") else record

## 1. Data

**Dataset:** [`zeroshot/twitter-financial-news-sentiment`](https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment)
on the Hugging Face Hub: English finance tweets, mostly headline-style, each labelled Bearish,
Bullish or Neutral. **Licence:** MIT, as stated on the dataset card. The Hub revision is
pinned in the code, so the data cannot change silently.

**Cleaning:** whitespace is normalised; empty and URL-only tweets are dropped; near-duplicates
(the same text after lower-casing and removing links, often one headline tweeted with
different short links) are removed so that copies cannot land in both train and test; groups
of near-duplicates with conflicting labels are dropped entirely.

**Split:** the Hub's two splits are pooled and re-split **once**, stratified by label with seed
42, into train / validation / test (70 / 15 / 15). Every model reads the same files, and the
test split is used only for the final evaluation. Because of the re-split, scores here are not
comparable with scores reported on the Hub's own validation split.

In [ ]:
report = read_result(data.split_report_path())
if report is None:
    print("not run yet: data/splits/split_report.json is missing")
else:
    cleaning = report["cleaning"]
    print(f"{report['dataset']} (revision {report['revision'][:8]}, licence {report['license']})")
    print(
        f"{cleaning['n_raw']:,} raw rows -> {cleaning['n_clean']:,} after removing "
        f"{cleaning['dropped_empty']} empty, {cleaning['dropped_url_only']} URL-only, "
        f"{cleaning['dropped_conflicting_labels']} conflicting-label and "
        f"{cleaning['dropped_duplicates']} near-duplicate rows"
    )
    print("Split sizes:", ", ".join(f"{name} {n:,}" for name, n in report["sizes"].items()))

Label counts and shares per split, counted from the split CSVs. Neutral is by far the largest
class, which is why **macro-F1** (the unweighted mean of the three per-class F1 scores) is the
main metric: a model that mostly predicts Neutral gets a high accuracy but a low macro-F1.

In [ ]:
try:
    splits = data.load_splits()
except FileNotFoundError:
    splits = None
    print("not run yet: the split CSVs in data/splits/ are missing")

if splits is not None:
    balance = data.class_balance(splits)  # one row per split: counts and % per label
    display(balance)

    fig, ax = plt.subplots(figsize=(7, 3))
    bar = 0.25  # bar thickness; three bars (train, val, test) per label
    for i, (split, colour) in enumerate(zip(balance.index, [BLUE, ORANGE, AQUA], strict=True)):
        shares = [balance.loc[split, f"{label}_%"] for label in config.LABELS]
        rows = [j + (i - 1) * (bar + 0.03) for j in range(len(config.LABELS))]
        n = balance.loc[split, "n"]
        ax.barh(rows, shares, height=bar, color=colour, label=f"{split} (n={n:,})")
    ax.set_yticks(range(len(config.LABELS)), config.LABELS)
    ax.invert_yaxis()
    ax.grid(axis="x")
    ax.set_xlabel("share of the split (%)")
    ax.set_title("Label shares per split")
    ax.legend(loc="upper right")
    plt.show()

    print("Two training examples per label:")
    for row in splits["train"].groupby("label").head(2).sort_values("label").itertuples():
        print(f"{config.ID2LABEL[row.label]:>8}: {row.text}")

## 2. Test-set results

One row per model, read from `results/metrics/<model>.json` (written by
`fnsl.metrics.save_results`); smoke runs, which use tiny data only to check the code, are
never shown. A model scored on fewer rows than the full test split is flagged
*not comparable*, and unparseable LLM answers count as wrong.

In [ ]:
records = [r for r in metrics.load_metrics() if not r.get("smoke")]
by_name = {r["model"]: r for r in records}
n_test_full = len(splits["test"]) if splits is not None else None

if records:
    rows = []
    for r in records:
        rows.append(
            {
                "model": r["model"],
                "description": reporting.MODEL_DESCRIPTIONS.get(r["model"], ""),
                "macro-F1": r["macro_f1"],
                "accuracy": r["accuracy"],
                **{f"F1 {label}": r["per_class_f1"][label] for label in config.LABELS},
                "n_test": r["n_test"],
                "unparseable": r.get("n_unparseable", 0),
                "train time (s)": r["train_time_s"],
                "commit": r["git_commit"],
                "note": reporting.comparable_note(r, n_test_full),
            }
        )
    table = pd.DataFrame(rows).sort_values("macro-F1", ascending=False, ignore_index=True)
    display(table.round(3))
missing = [name for name in reporting.EXPECTED_MODELS if name not in by_name]
if missing:
    print("not run yet:", ", ".join(missing))

## 3. Are the differences real?

The test set is a sample, so every score carries sampling noise. Two checks, both computed by
`scripts/compare_models.py` and stored in `results/metrics/comparison.json`:

- **Bootstrap confidence interval:** resample the test rows with replacement many times,
  recompute macro-F1 each time and keep the middle 95% of those scores. Overlapping intervals
  are a warning that two models may not really differ.
- **McNemar's test** (paired): uses only the test rows where exactly one of two models is
  right. `b` counts rows only model A gets right, `c` rows only model B gets right. If both
  models were equally good, b and c would differ only by chance; the p-value says how
  surprising the observed gap is. With many pairs, the Holm correction keeps the chance of any
  false alarm at the chosen level.

Only intervals and pairs computed from the same predictions as the metrics files above are
shown; anything older is reported as stale.

In [ ]:
comparison = read_result(config.METRICS_DIR / "comparison.json")
cis = {}
if comparison is None:
    print("not run yet: results/metrics/comparison.json is missing")
else:
    level = reporting.ci_level(comparison)
    print(f"{comparison['n_resamples']:,} bootstrap resamples, alpha = {comparison['alpha']}")
    # consistent_ci gives None when comparison.json was computed from older predictions.
    for name, record in by_name.items():
        ci = reporting.consistent_ci(name, record, comparison)
        if ci is not None:
            cis[name] = ci
    stale = sorted(set(comparison["models"]) - set(cis))
    if stale:
        print("stale (rerun scripts/compare_models.py), not shown:", ", ".join(stale))

if cis:
    names = sorted(cis, key=lambda name: cis[name]["macro_f1"])  # best model ends up on top
    f1 = [cis[name]["macro_f1"] for name in names]
    low = [cis[name]["ci_low"] for name in names]
    high = [cis[name]["ci_high"] for name in names]
    # errorbar() wants the distance from the point to each end of the interval.
    xerr = [
        [f - lo for f, lo in zip(f1, low, strict=True)],
        [hi - f for f, hi in zip(f1, high, strict=True)],
    ]

    fig, ax = plt.subplots(figsize=(7, 1.2 + 0.5 * len(names)))
    ax.errorbar(
        f1,
        range(len(names)),
        xerr=xerr,
        fmt="o",
        color=BLUE,
        elinewidth=2,
        markersize=8,
        markeredgecolor=SURFACE,  # thin ring in the background colour around each dot
        markeredgewidth=2,
    )
    for row in range(len(names)):
        label = f"{f1[row]:.3f} [{low[row]:.3f}, {high[row]:.3f}]"
        ax.annotate(label, (high[row], row), xytext=(8, 0), textcoords="offset points", va="center")
    ax.set_yticks(range(len(names)), names)
    ax.set_ylim(-0.6, len(names) - 0.4)
    ax.set_xlim(max(0.0, min(low) - 0.05), min(1.0, max(high) + 0.12))
    ax.grid(axis="x")
    ax.set_xlabel(f"test macro-F1 with {level}% bootstrap CI")
    ax.set_title("Macro-F1 per model")
    plt.show()

**McNemar's test for every pair** of models scored on the same test rows. A pair counts as
significant when its Holm-adjusted p-value is below alpha. The macro-F1 difference and its CI
come from a paired bootstrap (both models scored on the same resampled rows). The two checks
can disagree, because macro-F1 weights the rare classes more than the error rate does.

In [ ]:
if comparison is not None:
    pairs, stale_pairs = reporting.split_pairs(comparison, by_name)
    rows = []
    for pair in pairs:
        mc, diff = pair["mcnemar"], pair["macro_f1_diff"]
        rows.append(
            {
                "A": pair["model_a"],
                "B": pair["model_b"],
                "n": pair["n"],
                "b (only A right)": mc["b"],
                "c (only B right)": mc["c"],
                "McNemar p": reporting.fmt_p(mc["p_value"]),
                "Holm p": reporting.fmt_p(pair["p_holm"]),
                "macro-F1 A - B": round(diff["diff"], 3),
                f"{level}% CI of A - B": reporting.fmt_ci(diff["low"], diff["high"]),
                "significant (Holm)": "yes" if pair["significant_holm"] else "no",
            }
        )
    if rows:
        display(pd.DataFrame(rows))
    else:
        print("not run yet: no two models with predictions on the same test rows")
    for pair in stale_pairs:
        print(f"stale, not shown: {pair['model_a']} vs {pair['model_b']}")
    for pair in comparison.get("skipped_pairs", []):
        print(f"skipped: {pair['model_a']} vs {pair['model_b']}: {pair['reason']}")

## 4. Where do the models go wrong?

Confusion matrices on the test split (rows = true label, columns = predicted label), saved by
each training script in `results/figures/`.

In [ ]:
if not records:
    print("not run yet: no model results")
for record in records:
    png = config.FIGURES_DIR / f"{record['model']}_confusion.png"
    if png.exists():
        display(Image(filename=str(png), width=380))
    else:
        print(f"not found: results/figures/{png.name}")

## 5. Unsupervised view: K-Means on sentence embeddings

Tweets are embedded with the sentence-transformer `all-MiniLM-L6-v2` and clustered with
K-Means without looking at the labels. The **silhouette** score says how well separated the
clusters are (-1 to 1); **ARI** and **NMI** say how much the clusters agree with the sentiment
labels (ARI: 0 = chance level, 1 = identical). The 2-D picture uses a PCA written from scratch
with NumPy's SVD.

In [ ]:
kmeans = read_result(config.METRICS_DIR / "kmeans.json")
if kmeans is None:
    print("not run yet: results/metrics/kmeans.json is missing")
else:
    print(f"{kmeans['embedding_model']} on the {kmeans['split']} split, n = {kmeans['n']:,}")
    runs = [
        {
            "k": run["k"],
            "silhouette": run["silhouette"],
            "ARI": run["ari"],
            "NMI": run["nmi"],
            "purity": run.get("overall_purity"),
        }
        for run in kmeans["runs"]
    ]
    display(pd.DataFrame(runs).round(3))
    pca_png = config.FIGURES_DIR / "kmeans_pca.png"
    if pca_png.exists():
        display(Image(filename=str(pca_png), width=900))

## 6. RAG over central-bank statements

A small retrieval-augmented question-answering pipeline (LangChain + FAISS) over public FOMC
statements and ECB monetary-policy press releases; sources and licences are listed in
`data/rag_corpus/SOURCES.md`. The questions are written by hand, each answerable from exactly
one document. **hit@k** is the share of questions whose source document is among the top-k
retrieved chunks. It measures retrieval only, not whether an answer is correct.

In [ ]:
rag = read_result(config.METRICS_DIR / "rag_eval.json")
if rag is None:
    print("not run yet: results/metrics/rag_eval.json is missing")
else:
    print(
        f"{rag['n_questions']} questions, {rag['n_documents']} documents, "
        f"embeddings {rag['embedding_model']}, best chunk size {rag['best_chunk_size']}"
    )
    ks = rag.get("k_values", [1, 3])
    configs = []
    for cfg in rag["configs"]:
        row = {"chunk size": cfg["chunk_size"], "chunks": cfg["n_chunks"]}
        row.update({f"hit@{k}": cfg[f"hit@{k}"] for k in ks})
        configs.append(row)
    display(pd.DataFrame(configs).round(3))

## 7. Findings

> **To be written after Day 10**, quoting only numbers shown above.

- **Best model:** _..._
- **Is the gap significant?** _Which pairs differ after the Holm correction, and which do not._
- **Cost:** _Training time and hardware for each extra point of macro-F1._
- **Typical errors:** _Which labels get confused, from the confusion matrices._
- **Zero-shot vs LoRA:** _What fine-tuning changed for the LLM._

## 8. Limitations

- **One small dataset:** short English tweets from a single source; results may not transfer
  to longer articles, other languages or other periods.
- **Short texts:** most posts are a single headline, so context that would settle the
  sentiment is often missing.
- **One fixed split:** bootstrap intervals cover test-set sampling noise, not the variation
  from a different split; seed variation is measured only where extra seeds were run.
- **Label noise:** labels come from the dataset authors; conflicting near-duplicates were
  removed, other noise remains.
- **No market validation:** the labels are not checked against price moves, so nothing here
  says the predictions are useful for trading.
- **RAG is scored on retrieval only:** hit@k checks that the right document is retrieved, not
  that an answer is correct.

## 9. Reproduce

Everything runs from the repository: CPU parts on a laptop, GPU parts through
`notebooks/colab_runner.ipynb` on Colab or Kaggle.

```bash
pip install -r requirements.txt && pip install -e . --no-deps
python scripts/run_baseline.py          # TF-IDF + logistic regression
python scripts/run_kmeans.py            # K-Means + NumPy PCA (needs requirements-local.txt)
python scripts/train_keras.py           # Keras BiLSTM (needs requirements-local.txt)
python scripts/compare_models.py        # bootstrap CIs + McNemar for all models
python scripts/eval_rag.py              # RAG hit@1 / hit@3 (needs requirements-local.txt)
python scripts/make_results_table.py    # print every table
```